# STEP 1 — PubMed 논문 초록 수집 (재현용)

PubMed에서 mAb 안정성 관련 논문의 **제목·초록**을 모아 `step1_abstracts.csv` 로 저장합니다.

| | |
|---|---|
| **입력** | 없음 (PubMed 공개 API) |
| **출력** | `step1_abstracts.csv` (pmid, title, abstract, year, journal) |
| **필요** | 인터넷만 (OpenAI 키 불필요·무료) |
| **다음** | 이 CSV를 STEP 2에 넣습니다 |

> 💡 원본 연구는 126개 검색어로 62,281편을 모았습니다. 여기서는 **빠르게 따라하도록 작게** 설정했어요. `SEARCH_QUERIES` 와 `MAX_RESULTS_PER_QUERY` 를 키우면 규모를 늘릴 수 있습니다.

In [ ]:
!pip install -q requests xmltodict pandas
print('준비 완료')

In [ ]:
import requests, time, xmltodict, pandas as pd

# ── 설정 (여기만 바꾸면 됩니다) ───────────────────────────────
PUBMED_EMAIL = 'student@example.com'      # NCBI 권장: 아무 이메일
MAX_RESULTS_PER_QUERY = 40                # 쿼리당 최대 논문 수 (작게 시작)
SEARCH_QUERIES = [
    '"monoclonal antibody" AND "stability" AND "formulation"',
    '"monoclonal antibody" AND ("aggregation" OR "viscosity")',
    '"therapeutic antibody" AND "stability"',
]
print(f'검색어 {len(SEARCH_QUERIES)}개, 쿼리당 최대 {MAX_RESULTS_PER_QUERY}편')

In [ ]:
def search_pubmed(query, max_results=40):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    p = {'db':'pubmed','term':query,'retmax':max_results,'retmode':'json','email':PUBMED_EMAIL}
    r = requests.get(url, params=p, timeout=30); r.raise_for_status()
    d = r.json().get('esearchresult', {})
    return d.get('idlist', []), int(d.get('count', '0'))

def fetch_abstracts(pmids):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    arts = []
    for i in range(0, len(pmids), 200):
        batch = pmids[i:i+200]
        p = {'db':'pubmed','id':','.join(batch),'rettype':'xml','retmode':'xml','email':PUBMED_EMAIL}
        r = requests.get(url, params=p, timeout=60); r.raise_for_status()
        papers = xmltodict.parse(r.text).get('PubmedArticleSet', {}).get('PubmedArticle', [])
        if isinstance(papers, dict): papers = [papers]
        for a in papers:
            try:
                mc = a['MedlineCitation']
                pmid = mc['PMID']['#text'] if isinstance(mc['PMID'], dict) else mc['PMID']
                art = mc['Article']
                title = art.get('ArticleTitle', '')
                if isinstance(title, dict): title = title.get('#text', '')
                ab = art.get('Abstract', {}).get('AbstractText', '')
                if isinstance(ab, list):
                    parts = []
                    for x in ab:
                        if isinstance(x, dict):
                            lab = x.get('@Label', ''); txt = x.get('#text', '')
                            parts.append(f'{lab}: {txt}' if lab else txt)
                        else: parts.append(str(x))
                    ab = ' '.join(parts)
                elif isinstance(ab, dict): ab = ab.get('#text', '')
                year = art.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {}).get('Year', 'N/A')
                journal = art.get('Journal', {}).get('Title', 'N/A')
                if ab and str(ab) != 'None' and len(str(ab)) > 50:
                    arts.append({'pmid':str(pmid),'title':str(title),'abstract':str(ab),'year':year,'journal':journal})
            except (KeyError, TypeError):
                continue
        time.sleep(0.4)   # NCBI rate limit 준수
    return arts

print('함수 정의 완료')

In [ ]:
all_pmids = set()
for i, q in enumerate(SEARCH_QUERIES, 1):
    pmids, total = search_pubmed(q, MAX_RESULTS_PER_QUERY)
    all_pmids.update(pmids)
    print(f'[{i}/{len(SEARCH_QUERIES)}] {len(pmids)}건 (전체 {total}건) | {q[:50]}')
    time.sleep(0.4)
print(f'\n중복 제거 후 {len(all_pmids)}편 → 초록 다운로드...')

articles = fetch_abstracts(list(all_pmids))
df = pd.DataFrame(articles)
df.to_csv('step1_abstracts.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ 저장: step1_abstracts.csv  ({len(df)}편)')
df.head(3)

### ✅ STEP 1 완료
`step1_abstracts.csv` 가 만들어졌습니다.

- **Colab:** 왼쪽 파일창에서 이 CSV를 다운로드해 두거나, STEP 2를 같은 런타임에서 이어 실행하세요.
- **다음:** `step2_filter.ipynb` 를 열어 이 CSV를 업로드(또는 같은 런타임 사용)합니다.